Copyright 2026 DataRobot, Inc. and its affiliates.

Licensed under the Apache License, Version 2.0 (the "License");
you may not use this file except in compliance with the License.
You may obtain a copy of the License at

   http://www.apache.org/licenses/LICENSE-2.0

Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS,
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
See the License for the specific language governing permissions and
limitations under the License.

In [ ]:
from jointfm_client import bootstrap_notebook

bootstrap_notebook(add_src_root=True)

# Conditional Forecast
Forecast USD portfolio NAV and risk from 100 positive float daily observations: equity index level, 10-year Treasury yield, and one EUR/USD FX rate. Instead of the unconditional forecast, ask the model a *what-if* question about the first future step: what does it expect for portfolio NAV and realized volatility **given** something about the other columns at that same step.

The `condition` query mode answers this in closed form from the model's joint distribution at one future position. A request names that position once, by its index into `query_times`, and attaches one condition per column it wants to fix:

- An **equality condition** pins a column to a value (`EqualityCondition`). The pinned column leaves the read-out set, because reading it back would only repeat the request.
- An **interval condition** confines a column to a range whose bounds may be open on either side (`IntervalCondition`). The column stays readable: what comes back is its distribution inside the range.

Every column without a condition is a read-out column, and the response describes the conditional distribution of those columns at the conditioned position only, so `outputs.query_times` has exactly one entry however many `query_times` the request carried.

Whether a deployment can condition depends on the checkpoint's head, so `/healthz` advertises `condition` in `supported_query_modes` and the kinds it answers in `supported_condition_kinds`. The client checks that advertisement before sending, and this notebook reads it explicitly so the check is visible.

In [ ]:
from pathlib import Path

import pandas as pd

from jointfm_client import (
    ConditionBlock,
    EqualityCondition,
    JointFMClient,
    plan_forecast_columns,
)

HISTORY_PATH = Path("notebooks/history.csv")
FEATURE_COLUMNS = ["equity_index_level", "treasury_10y_yield", "eur_usd_rate"]
TARGET_COLUMNS = ["portfolio_nav", "realized_volatility"]
INPUT_STEPS = 100
OUTPUT_HORIZONS = 10
CONDITIONED_STEP = 0
EQUITY_RALLY = 1.02
EXPECTED_COLUMNS = FEATURE_COLUMNS + TARGET_COLUMNS
QUERY_TIMES = list(range(INPUT_STEPS, INPUT_STEPS + OUTPUT_HORIZONS))

history = pd.read_csv(HISTORY_PATH, dtype=float)
if list(history.columns) != EXPECTED_COLUMNS:
    raise ValueError(
        f"Expected columns {EXPECTED_COLUMNS!r}, got {list(history.columns)!r}"
    )
if len(history) != INPUT_STEPS:
    raise ValueError(f"Expected {INPUT_STEPS} history rows, got {len(history)}")

client = JointFMClient.from_env()
health = client.health(cache=True)
if "condition" not in health.supported_query_modes:
    raise RuntimeError(
        f"This deployment serves {list(health.supported_query_modes)} only; "
        "mount a checkpoint whose head can condition"
    )
print("condition kinds served:", list(health.supported_condition_kinds))

plan = plan_forecast_columns(
    health=health,
    feature_columns=FEATURE_COLUMNS,
    target_columns=TARGET_COLUMNS,
    history_length=len(history),
    query_times_length=len(QUERY_TIMES),
)

last_equity_level = float(history["equity_index_level"].iloc[-1])
rally = ConditionBlock(
    query_time_index=CONDITIONED_STEP,
    conditions=[
        EqualityCondition(
            column="equity_index_level", value=last_equity_level * EQUITY_RALLY
        )
    ],
)
result = client.forecast_mean(
    history,
    query_times=QUERY_TIMES,
    requested_columns=plan.requested_columns,
    columns=plan.columns,
    seed=7,
    condition=rally,
)
if result.query_times != (QUERY_TIMES[CONDITIONED_STEP],):
    raise ValueError(
        f"Expected the conditioned position alone, got {result.query_times!r}"
    )
if result.plausibility is None:
    raise ValueError("A condition response must carry its plausibility block")
print("log density of the pinned value:", result.plausibility.equality_log_density)
forecast = result.to_pandas_tidy()
expected_forecast_rows = len(plan.requested_columns)
if len(forecast) != expected_forecast_rows:
    raise ValueError(
        f"Expected {expected_forecast_rows} forecast rows, got {len(forecast)}"
    )
forecast

## Reading the plausibility
`result.plausibility.equality_log_density` is the log density the model assigns to the pinned value before conditioning. It separates *the model is confident about NAV given this rally* from *the model finds a rally of this size absurd and is extrapolating*. The service reports the number and never refuses on it; comparing it across candidate pins, or against the density of a pin at the model's own unconditional mean, is the caller's decision.

## Interval condition
Now confine the 10-year yield to a band around its last observed value instead of pinning it, and pin the equity index at the same time: a request may mix both kinds across the columns of one position. The response then carries `region_log_probability`, the log probability the model gives the yield band, and the yield column itself stays readable because its distribution inside the band is a genuine answer.

With one interval column the region probability is exact. When several columns carry intervals the service estimates the probability of the box numerically and reports the accounting in `diagnostics.interval_estimator`, so the caller can judge the estimate.

In [ ]:
from jointfm_client import IntervalCondition

YIELD_BAND_HALF_WIDTH = 0.002

last_yield = float(history["treasury_10y_yield"].iloc[-1])
rally_with_yield_band = ConditionBlock(
    query_time_index=CONDITIONED_STEP,
    conditions=[
        EqualityCondition(
            column="equity_index_level", value=last_equity_level * EQUITY_RALLY
        ),
        IntervalCondition(
            column="treasury_10y_yield",
            lower=last_yield - YIELD_BAND_HALF_WIDTH,
            upper=last_yield + YIELD_BAND_HALF_WIDTH,
        ),
    ],
)
banded = client.forecast_mean(
    history,
    query_times=QUERY_TIMES,
    requested_columns=["treasury_10y_yield", *plan.requested_columns],
    columns=plan.columns,
    seed=7,
    condition=rally_with_yield_band,
)
if banded.plausibility is None:
    raise ValueError("A condition response must carry its plausibility block")
print("log density of the pinned value:", banded.plausibility.equality_log_density)
print("log probability of the yield band:", banded.plausibility.region_log_probability)
print("interval estimator accounting:", banded.diagnostics.interval_estimator)
banded.to_pandas_tidy()